<a href="https://colab.research.google.com/github/nieblasIX/IX-SierraGorda-Monitoring/blob/main/01_SierraGorda_Degradacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Instalamos la herramienta para ver mapas (geemap)
!pip install geemap

# 2. Llamamos a Google Earth Engine
import ee
import geemap

# 3. Pedimos permiso para usar tu cuenta (Autenticación)
ee.Authenticate()
ee.Initialize()
# OJO: Si no tienes un proyecto de Google Cloud, borra lo de adentro del paréntesis y deja solo: ee.Initialize()

In [135]:
# Reemplaza 'TU-PROYECTO-AQUI' con el ID real de tu proyecto de Google Cloud
proyecto_id = 'sierragorda-2026'

ee.Initialize(project=proyecto_id)

# Creamos el mapa interactivo
Map = geemap.Map()

# Nombres de los municipios
municipios = ['Pisaflores', 'La Mision', 'Jacala De Ledezma', 'Pacula', 'Zimapan']

# Buscamos México y filtramos la Sierra Gorda
mexico = ee.FeatureCollection("FAO/GAUL/2015/level2")
sierra_gorda = mexico.filter(ee.Filter.inList('ADM2_NAME', municipios))

# Centramos el mapa y lo pintamos de rojo
Map.centerObject(sierra_gorda, 10)
Map.addLayer(sierra_gorda, {'color': 'red'}, 'Sierra Gorda Hidalguense')

# Mostramos el mapa
Map

Map(center=[20.936798264152298, -99.24585161971991], controls=(WidgetControl(options=['position', 'transparent…

In [134]:
# Usamos satélites Landsat 8 de los últimos años
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(sierra_gorda) \
    .filterDate('2018-01-01', '2023-12-31')

# Función matemática (Índice NBR: detecta estrés hídrico y biomasa)
def calcular_salud(image):
    nbr = image.normalizedDifference(['SR_B5', 'SR_B7']).rename('Salud_Forestal')
    return image.addBands(nbr)

# Aplicamos la función y sacamos el promedio
l8_salud = l8.map(calcular_salud)
promedio_salud = l8_salud.select('Salud_Forestal').mean().clip(sierra_gorda)

# NUEVA PALETA DE ALTO CONTRASTE (Cartografía Profesional)
# Rojo oscuro -> Naranja -> Amarillo -> Verde claro -> Verde esmeralda
colores_salud = {
    'min': -0.1,
    'max': 0.6,
    'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
}

# Añadimos al mapa
Map.addLayer(promedio_salud, colores_salud, 'Salud Forestal (Alto Contraste)')
Map

Map(bottom=115776.0, center=[20.93578924489374, -99.24705505371094], controls=(WidgetControl(options=['positio…